# Face Cropping Preprocessing

This notebook applies face detection and face cropping before model training.

The goal is to convert raw full images into cropped-face images for KMU-FED, KDEF, and FER2013.  
The cropped images will later be used to train and evaluate the SqueezeNet model.

Pipeline:

1. Load raw manifest CSV files
2. Detect face using MTCNN
3. Crop the detected face region
4. Save cropped-face images
5. Create new cropped manifest CSV files

In [ ]:
!pip install facenet-pytorch --no-deps

In [ ]:
# Google Drive
from google.colab import drive

# File and path handling
from pathlib import Path
import os

# Data handling
import pandas as pd
import numpy as np

# Image handling
from PIL import Image, ImageOps

# Visualization
import matplotlib.pyplot as plt

# Progress bar
from tqdm.auto import tqdm

# PyTorch
import torch

# MTCNN face detector
from facenet_pytorch import MTCNN


In [ ]:
drive.mount("/content/drive")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

## Project Paths

This section defines the input raw manifest directory and the output directories for cropped-face images and cropped manifest CSV files.

In [ ]:
# Main project directory in Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/FER_Generalization_Project")

# Input raw manifest directory from Notebook 01
RAW_MANIFEST_DIR = PROJECT_DIR / "data_processed" / "manifests"

# Output directories for this notebook
CROPPED_IMAGE_DIR = PROJECT_DIR / "data_processed" / "cropped_faces"
CROPPED_MANIFEST_DIR = PROJECT_DIR / "data_processed" / "manifests_cropped"

# Create output directories if they do not exist
CROPPED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
CROPPED_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("Raw manifest directory:", RAW_MANIFEST_DIR)
print("Raw manifest directory exists:", RAW_MANIFEST_DIR.exists())
print("Cropped image directory:", CROPPED_IMAGE_DIR)
print("Cropped manifest directory:", CROPPED_MANIFEST_DIR)

## Load Raw Manifest Files

The raw manifest CSV files created in Notebook 01 are loaded here.  
These files contain the paths to the original raw images and their aligned emotion labels.

In [ ]:
# Load raw manifest files
kmu_train_df = pd.read_csv(RAW_MANIFEST_DIR / "kmu_train.csv")
kmu_test_df = pd.read_csv(RAW_MANIFEST_DIR / "kmu_test.csv")
kdef_test_df = pd.read_csv(RAW_MANIFEST_DIR / "kdef_frontal_test.csv")
fer2013_test_df = pd.read_csv(RAW_MANIFEST_DIR / "fer2013_test.csv")

print("KMU train:", kmu_train_df.shape)
print("KMU test:", kmu_test_df.shape)
print("KDEF test:", kdef_test_df.shape)
print("FER2013 test:", fer2013_test_df.shape)

kmu_train_df.head()

## Initialize MTCNN Face Detector

MTCNN is used to detect the face area in each raw image.  
After detection, the face region will be cropped, resized, and saved as a new image.

In [ ]:
# Initialize MTCNN face detector
mtcnn = MTCNN(
    keep_all=True,
    device=device,
    min_face_size=20
)

print("MTCNN initialized on:", device)

## Face Cropping Helper Functions

These helper functions detect the face, crop it with a small margin, resize it to 224×224, and return the cropped image.

If no face is detected, a fallback center crop is used so that the image is not lost.

In [ ]:
def get_path_column(df):
    """
    Detects the image path column name in the manifest.
    """
    if "path" in df.columns:
        return "path"
    elif "image_path" in df.columns:
        return "image_path"
    else:
        raise ValueError("No image path column found. Expected 'path' or 'image_path'.")


def center_square_crop(image):
    """
    Fallback crop used when no face is detected.
    It crops the center square region of the image.
    """
    width, height = image.size
    min_side = min(width, height)

    left = (width - min_side) // 2
    top = (height - min_side) // 2
    right = left + min_side
    bottom = top + min_side

    return image.crop((left, top, right, bottom))


def detect_and_crop_face(image_path, output_size=(224, 224), margin_ratio=0.20):
    """
    Detects and crops the largest face from an image using MTCNN.

    Returns:
    - cropped image
    - face detection status
    - detection probability
    """

    image = Image.open(image_path)
    image = ImageOps.exif_transpose(image)
    image = image.convert("RGB")

    boxes, probs = mtcnn.detect(image)

    # If no face is detected, use fallback center crop
    if boxes is None or len(boxes) == 0:
        cropped = center_square_crop(image)
        cropped = cropped.resize(output_size, Image.Resampling.BILINEAR)
        return cropped, "fallback_center_crop", None

    # Select the largest detected face
    areas = []
    for box in boxes:
        x1, y1, x2, y2 = box
        area = (x2 - x1) * (y2 - y1)
        areas.append(area)

    largest_idx = int(np.argmax(areas))
    x1, y1, x2, y2 = boxes[largest_idx]
    prob = probs[largest_idx] if probs is not None else None

    # Add margin around face
    width, height = image.size
    box_width = x2 - x1
    box_height = y2 - y1

    margin_x = box_width * margin_ratio
    margin_y = box_height * margin_ratio

    x1 = max(0, int(x1 - margin_x))
    y1 = max(0, int(y1 - margin_y))
    x2 = min(width, int(x2 + margin_x))
    y2 = min(height, int(y2 + margin_y))

    cropped = image.crop((x1, y1, x2, y2))
    cropped = cropped.resize(output_size, Image.Resampling.BILINEAR)

    return cropped, "face_detected", prob

## Test Face Cropping on One KMU-FED Image

Before processing all images, we test face cropping on one KMU-FED image.

In [ ]:
path_col = get_path_column(kmu_train_df)

sample_row = kmu_train_df.sample(1, random_state=42).iloc[0]
sample_path = sample_row[path_col]
sample_label = sample_row["label"]

raw_image = Image.open(sample_path)
raw_image = ImageOps.exif_transpose(raw_image)
raw_image = raw_image.convert("RGB")
cropped_image, status, prob = detect_and_crop_face(sample_path)

print("Image path:", sample_path)
print("Label:", sample_label)
print("Cropping status:", status)
print("Detection probability:", prob)

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(raw_image)
plt.title("Raw Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cropped_image)
plt.title("Cropped Face")
plt.axis("off")

plt.show()

## Test Face Cropping on Multiple KMU-FED Samples

This visual check helps confirm that MTCNN is cropping the face correctly before applying it to the full dataset.

In [ ]:
sample_df = kmu_train_df.sample(8, random_state=7).reset_index(drop=True)

plt.figure(figsize=(12, 8))

for i, row in sample_df.iterrows():
    image_path = row[path_col]
    label = row["label"]

    cropped_image, status, prob = detect_and_crop_face(image_path)

    plt.subplot(2, 4, i + 1)
    plt.imshow(cropped_image)
    plt.title(f"{label}\n{status}")
    plt.axis("off")

plt.suptitle("KMU-FED Cropping Test Samples", fontsize=16)
plt.tight_layout()
plt.show()

## Process and Save Cropped-Face Images

This section applies MTCNN face detection and cropping to each image listed in the raw manifest files.

For every image, the notebook saves:

- the cropped-face image
- face detection status
- detection probability
- a new cropped manifest CSV file

If MTCNN does not detect a face, a fallback center crop is used so that the image is not lost.

In [ ]:
def process_manifest(df, output_dataset_name, output_manifest_name, force_reprocess=False):
    """
    Processes one manifest DataFrame:
    - reads raw images
    - detects and crops faces
    - saves cropped images
    - creates a new cropped manifest CSV

    If the cropped manifest already exists, it is loaded directly unless force_reprocess=True.
    """

    output_csv_path = CROPPED_MANIFEST_DIR / output_manifest_name

    # If already processed, load existing manifest instead of cropping again
    if output_csv_path.exists() and not force_reprocess:
        print(f"Existing cropped manifest found: {output_csv_path}")
        print("Loading existing manifest instead of reprocessing images.")
        return pd.read_csv(output_csv_path)

    path_col = get_path_column(df)

    processed_rows = []

    output_dataset_dir = CROPPED_IMAGE_DIR / output_dataset_name
    output_dataset_dir.mkdir(parents=True, exist_ok=True)

    print(f"Processing: {output_manifest_name}")
    print(f"Total images: {len(df)}")
    print("Saving cropped images to:", output_dataset_dir)

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        original_path = row[path_col]
        label = row["label"]

        label_dir = output_dataset_dir / label
        label_dir.mkdir(parents=True, exist_ok=True)

        original_path_obj = Path(original_path)

        # Unique output filename
        output_filename = f"{idx:06d}_{original_path_obj.stem}.jpg"
        cropped_path = label_dir / output_filename

        try:
            cropped_image, status, prob = detect_and_crop_face(original_path)
            cropped_image.save(cropped_path)

        except Exception as e:
            status = "error"
            prob = None
            cropped_path = None
            print(f"Error at index {idx}: {original_path}")
            print(e)

        new_row = row.to_dict()
        new_row["original_path"] = original_path
        new_row["cropped_path"] = str(cropped_path) if cropped_path is not None else None
        new_row["face_crop_status"] = status
        new_row["face_detection_probability"] = prob

        processed_rows.append(new_row)

    cropped_df = pd.DataFrame(processed_rows)

    # Remove rows where image processing completely failed
    before_count = len(cropped_df)
    cropped_df = cropped_df[cropped_df["cropped_path"].notna()].reset_index(drop=True)
    after_count = len(cropped_df)

    cropped_df.to_csv(output_csv_path, index=False)

    print("\nDone:", output_manifest_name)
    print("Before:", before_count)
    print("After:", after_count)
    print("Saved manifest:", output_csv_path)

    print("\nFace crop status counts:")
    print(cropped_df["face_crop_status"].value_counts())

    return cropped_df

## Process KMU-FED Training Set First

We start with KMU-FED training images because this is the source dataset for model training.

In [ ]:
kmu_train_cropped_df = process_manifest(
    df=kmu_train_df,
    output_dataset_name="KMU_FED/train",
    output_manifest_name="kmu_train_cropped.csv"
)

In [ ]:
print(kmu_train_cropped_df.shape)
display(kmu_train_cropped_df.head())

print("\nLabel counts:")
print(kmu_train_cropped_df["label"].value_counts())

print("\nCrop status counts:")
print(kmu_train_cropped_df["face_crop_status"].value_counts())

## Visual Check After Saving Cropped Images

This confirms that the saved cropped-face images can be loaded correctly from disk.

In [ ]:
sample_saved_df = kmu_train_cropped_df.sample(8, random_state=10).reset_index(drop=True)

plt.figure(figsize=(12, 8))

for i, row in sample_saved_df.iterrows():
    img = Image.open(row["cropped_path"]).convert("RGB")

    plt.subplot(2, 4, i + 1)
    plt.imshow(img)
    plt.title(f"{row['label']}\n{row['face_crop_status']}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Check KMU-FED Fallback Crops

This section inspects images where MTCNN did not detect a face and fallback center cropping was used.

The purpose is to confirm that fallback crops still contain useful facial information.

In [ ]:
fallback_df = kmu_train_cropped_df[
    kmu_train_cropped_df["face_crop_status"] == "fallback_center_crop"
].reset_index(drop=True)

print("Number of fallback images:", len(fallback_df))

display(fallback_df.head())

if len(fallback_df) > 0:
    sample_fallback_df = fallback_df.sample(
        min(8, len(fallback_df)),
        random_state=42
    ).reset_index(drop=True)

    plt.figure(figsize=(12, 8))

    for i, row in sample_fallback_df.iterrows():
        img = Image.open(row["cropped_path"])
        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")

        plt.subplot(2, 4, i + 1)
        plt.imshow(img)
        plt.title(f"{row['label']}\n{row['face_crop_status']}")
        plt.axis("off")

    plt.suptitle("KMU-FED Fallback Center Crop Samples", fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("No fallback images found.")

## Process KMU-FED Test Set

This section applies the same face cropping process to the KMU-FED test set.

The KMU-FED test set will later be used to evaluate the baseline model trained on KMU-FED training images.

In [ ]:
kmu_test_cropped_df = process_manifest(
    df=kmu_test_df,
    output_dataset_name="KMU_FED/test",
    output_manifest_name="kmu_test_cropped.csv"
)

In [ ]:
print(kmu_test_cropped_df.shape)

print("\nLabel counts:")
print(kmu_test_cropped_df["label"].value_counts())

print("\nCrop status counts:")
print(kmu_test_cropped_df["face_crop_status"].value_counts())

In [ ]:
kdef_test_cropped_df = process_manifest(
    df=kdef_test_df,
    output_dataset_name="KDEF/test",
    output_manifest_name="kdef_frontal_test_cropped.csv"
)

In [ ]:
print(kdef_test_cropped_df.shape)

print("\nLabel counts:")
print(kdef_test_cropped_df["label"].value_counts())

print("\nCrop status counts:")
print(kdef_test_cropped_df["face_crop_status"].value_counts())

In [ ]:
fer2013_test_cropped_df = process_manifest(
    df=fer2013_test_df,
    output_dataset_name="FER2013/test",
    output_manifest_name="fer2013_test_cropped.csv"
)

In [ ]:
print(fer2013_test_cropped_df.shape)

print("\nLabel counts:")
print(fer2013_test_cropped_df["label"].value_counts())

print("\nCrop status counts:")
print(fer2013_test_cropped_df["face_crop_status"].value_counts())

In [ ]:
summary_data = {
    "manifest": [
        "kmu_train_cropped",
        "kmu_test_cropped",
        "kdef_frontal_test_cropped",
        "fer2013_test_cropped"
    ],
    "total_images": [
        len(kmu_train_cropped_df),
        len(kmu_test_cropped_df),
        len(kdef_test_cropped_df),
        len(fer2013_test_cropped_df)
    ],
    "face_detected": [
        (kmu_train_cropped_df["face_crop_status"] == "face_detected").sum(),
        (kmu_test_cropped_df["face_crop_status"] == "face_detected").sum(),
        (kdef_test_cropped_df["face_crop_status"] == "face_detected").sum(),
        (fer2013_test_cropped_df["face_crop_status"] == "face_detected").sum()
    ],
    "fallback_center_crop": [
        (kmu_train_cropped_df["face_crop_status"] == "fallback_center_crop").sum(),
        (kmu_test_cropped_df["face_crop_status"] == "fallback_center_crop").sum(),
        (kdef_test_cropped_df["face_crop_status"] == "fallback_center_crop").sum(),
        (fer2013_test_cropped_df["face_crop_status"] == "fallback_center_crop").sum()
    ]
}

cropping_summary_df = pd.DataFrame(summary_data)
display(cropping_summary_df)

summary_path = CROPPED_MANIFEST_DIR / "cropping_summary.csv"
cropping_summary_df.to_csv(summary_path, index=False)

print("Saved cropping summary to:", summary_path)

In [ ]:
def display_cropped_samples(df, title, n=8, random_state=42):
    sample_df = df.sample(min(n, len(df)), random_state=random_state).reset_index(drop=True)

    plt.figure(figsize=(12, 8))

    for i, row in sample_df.iterrows():
        img = Image.open(row["cropped_path"]).convert("RGB")

        plt.subplot(2, 4, i + 1)
        plt.imshow(img)
        plt.title(f"{row['label']}\n{row['face_crop_status']}")
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
display_cropped_samples(kmu_train_cropped_df, "KMU-FED Train Cropped Samples")
display_cropped_samples(kmu_test_cropped_df, "KMU-FED Test Cropped Samples")
display_cropped_samples(kdef_test_cropped_df, "KDEF Test Cropped Samples")
display_cropped_samples(fer2013_test_cropped_df, "FER2013 Test Cropped Samples")

In [ ]:
fer2013_fallback_df = fer2013_test_cropped_df[
    fer2013_test_cropped_df["face_crop_status"] == "fallback_center_crop"
].reset_index(drop=True)

print("FER2013 fallback images:", len(fer2013_fallback_df))

display_cropped_samples(
    fer2013_fallback_df,
    "FER2013 Fallback Center Crop Samples",
    n=8,
    random_state=20
)

## Final Summary

Face detection and face cropping were applied to KMU-FED, KDEF, and FER2013 using MTCNN.

The cropped-face images were saved into:

`data_processed/cropped_faces/`

New cropped manifest CSV files were saved into:

`data_processed/manifests_cropped/`

The generated cropped manifests are:

- `kmu_train_cropped.csv`
- `kmu_test_cropped.csv`
- `kdef_frontal_test_cropped.csv`
- `fer2013_test_cropped.csv`
- `cropping_summary.csv`

For images where MTCNN could not detect a face, a fallback center crop was used. This prevents image loss and keeps the dataset size consistent.

The next step is `03_kmu_fed_baseline_training.ipynb`, where SqueezeNet 1.1 will be trained on KMU-FED cropped-face images and evaluated on KMU-FED test cropped-face images.